# ACV Satellite Monitoring

Classificação de imagens de satélite para monitoramento de queimadas florestais.

# 1. Introdução

Este projeto utiliza o dataset Sen2Fire para classificação de imagens de satélite contendo áreas com queimadas florestais.

Objetivos:

- Explorar a estrutura do dataset.
- Construir um dataset de classificação binária.
- Treinar diferentes arquiteturas CNN.
- Comparar os resultados obtidos.

In [ ]:
import os
import random
from pathlib import Path
from copy import deepcopy
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_recall_fscore_support,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

# 2. Configuração do Projeto

Definição dos diretórios utilizados durante o desenvolvimento.

In [ ]:
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw dir:", RAW_DIR)
print("Processed dir:", PROCESSED_DIR)
print("Results dir:", RESULTS_DIR)
print("Artifacts dir:", ARTIFACTS_DIR)

# 3. Exploração Inicial do Dataset

Nesta etapa é realizada uma inspeção da estrutura dos arquivos do dataset Sen2Fire.

In [ ]:
SEN2FIRE_DIR = RAW_DIR / "sen2fire"
print(SEN2FIRE_DIR)
print(list(SEN2FIRE_DIR.iterdir())[:10])

npz_files = sorted(SEN2FIRE_DIR.rglob("*.npz"))
print("Total de arquivos:", len(npz_files))
print("Primeiros arquivos:", npz_files[:5])

In [ ]:
sample = np.load(npz_files[0], allow_pickle=True)
print(sample.files)

for key in sample.files:
    arr = sample[key]
    print(f"{key}: type={type(arr)}, shape={arr.shape}, dtype={arr.dtype}")

image = sample["image"]
aerosol = sample["aerosol"]
label = sample["label"]

print("image min/max:", image.min(), image.max())
print("aerosol min/max:", aerosol.min(), aerosol.max())
print("Valores únicos no label:", np.unique(label))
print("Quantidade de pixels > 0:", (label > 0).sum())
print("Total de pixels:", label.size)
print("Percentual de pixels com fogo:", (label > 0).sum() / label.size * 100)

In [ ]:
fire_flags = []
scene_counts = Counter()

for file_path in npz_files:
    with np.load(file_path, allow_pickle=True) as data:
        lbl = data["label"]
        has_fire = int((lbl > 0).any())
        fire_flags.append(has_fire)

        scene = file_path.parent.name
        scene_counts[(scene, has_fire)] += 1

fire_flags = np.array(fire_flags)

df = pd.DataFrame({
    "file_path": [str(p) for p in npz_files],
    "has_fire": fire_flags,
    "scene": [p.parent.name for p in npz_files],
})

df["patch_label"] = df["has_fire"].astype(int)

scene_df = pd.DataFrame(
    [(scene, fire, count) for (scene, fire), count in scene_counts.items()],
    columns=["scene", "has_fire", "count"]
).sort_values(["scene", "has_fire"])

print("Total de patches:", len(fire_flags))
print("Patches com fogo:", fire_flags.sum())
print("Patches sem fogo:", (fire_flags == 0).sum())
print("Percentual com fogo:", fire_flags.mean() * 100)

print("\nDataFrame:")
print(df.shape)
print(df["patch_label"].value_counts())

print("\nPor cena:")
print(scene_df)

# 4. Visualização dos Dados

Exemplos de patches contendo fogo e sem fogo são visualizados para compreender o problema.

In [ ]:
def normalize_for_display(band):
    band = band.astype(np.float32)
    p2, p98 = np.percentile(band, (2, 98))

    if p98 - p2 < 1e-6:
        return np.zeros_like(band, dtype=np.float32)

    band = (band - p2) / (p98 - p2)
    return np.clip(band, 0, 1).astype(np.float32)

def make_rgb(image, r=3, g=2, b=1):
    rgb = np.stack([
        normalize_for_display(image[r]),
        normalize_for_display(image[g]),
        normalize_for_display(image[b]),
    ], axis=-1)
    return rgb

def load_npz(file_path):
    data = np.load(file_path, allow_pickle=True)
    return data["image"], data["aerosol"], data["label"]

sample_fire = df[df["patch_label"] == 1].iloc[0]
sample_nofire = df[df["patch_label"] == 0].iloc[0]

img_fire, aerosol_fire, label_fire = load_npz(Path(sample_fire["file_path"]))
img_nofire, aerosol_nofire, label_nofire = load_npz(Path(sample_nofire["file_path"]))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(make_rgb(img_fire))
axes[0, 0].set_title("Fire - RGB")
axes[0, 1].imshow(img_fire[0], cmap="gray")
axes[0, 1].set_title("Fire - banda 0")
axes[0, 2].imshow(label_fire, cmap="gray")
axes[0, 2].set_title("Fire - label")

axes[1, 0].imshow(make_rgb(img_nofire))
axes[1, 0].set_title("No fire - RGB")
axes[1, 1].imshow(img_nofire[0], cmap="gray")
axes[1, 1].set_title("No fire - banda 0")
axes[1, 2].imshow(label_nofire, cmap="gray")
axes[1, 2].set_title("No fire - label")

for ax in axes.ravel():
    ax.axis("off")

plt.tight_layout()
plt.show()

# 5. Construção do Dataset de Classificação

O rótulo por patch é binário:

- 0 = sem fogo
- 1 = com fogo

In [ ]:
train_df = df[df["scene"].isin(["scene1", "scene2"])].copy()
val_df = df[df["scene"] == "scene3"].copy()
test_df = df[df["scene"] == "scene4"].copy()

print("Treino:", train_df.shape)
print("Validação:", val_df.shape)
print("Teste:", test_df.shape)

print("\nTreino:")
print(train_df["patch_label"].value_counts())

print("\nValidação:")
print(val_df["patch_label"].value_counts())

print("\nTeste:")
print(test_df["patch_label"].value_counts())

# 6. Preparação para Treinamento

Criação do Dataset PyTorch, DataLoaders e tratamento do desbalanceamento.

In [ ]:
IMG_SIZE = 192
BATCH_SIZE = 8

def random_geometric_augment(x):
    if random.random() < 0.5:
        x = torch.flip(x, dims=[2])
    if random.random() < 0.5:
        x = torch.flip(x, dims=[1])
    k = random.randint(0, 3)
    x = torch.rot90(x, k, dims=[1, 2])
    return x

class Sen2FireDataset(Dataset):
    def __init__(self, dataframe, img_size=192, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.img_size = img_size
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = Path(row["file_path"])

        with np.load(file_path, allow_pickle=True) as data:
            image = data["image"]
            label = data["label"]

        y = 1 if (label > 0).any() else 0

        x = torch.tensor(image, dtype=torch.float32)
        x = F.interpolate(
            x.unsqueeze(0),
            size=(self.img_size, self.img_size),
            mode="bilinear",
            align_corners=False
        ).squeeze(0)

        if self.augment:
            x = random_geometric_augment(x)

        x = (x - x.mean()) / (x.std() + 1e-6)
        return x, torch.tensor(y, dtype=torch.long)

train_dataset = Sen2FireDataset(train_df, img_size=IMG_SIZE, augment=True)
val_dataset = Sen2FireDataset(val_df, img_size=IMG_SIZE, augment=False)
test_dataset = Sen2FireDataset(test_df, img_size=IMG_SIZE, augment=False)

x, y = train_dataset[0]
print(x.shape, x.dtype)
print(y)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

batch_x, batch_y = next(iter(train_loader))
print(batch_x.shape)
print(batch_y.shape)

class_counts = train_df["patch_label"].value_counts().sort_index()
class_weights = 1.0 / torch.tensor(class_counts.values, dtype=torch.float32)
class_weights = class_weights / class_weights.sum() * len(class_weights)

print("\nClasses no treino:")
print(class_counts)

print("\nPesos:")
print(class_weights)

# 7. Modelo 1 — BaselineCNN

Primeira arquitetura utilizada como baseline para comparação.

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, in_channels=12, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = BaselineCNN().to(device)
model

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    running_correct = 0

    for X, y in loader:
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X.size(0)
        running_correct += (torch.argmax(logits, dim=1) == y).sum().item()

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = running_correct / len(loader.dataset)
    return epoch_loss, epoch_acc

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_correct = 0
    all_targets = []
    all_preds = []

    for X, y in loader:
        X = X.to(device)
        y = y.to(device)

        logits = model(X)
        loss = criterion(logits, y)
        preds = torch.argmax(logits, dim=1)

        running_loss += loss.item() * X.size(0)
        running_correct += (preds == y).sum().item()

        all_targets.extend(y.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = running_correct / len(loader.dataset)
    return epoch_loss, epoch_acc, np.array(all_targets), np.array(all_preds)

def fit_model(model, train_loader, val_loader, criterion, optimizer, epochs=20, scheduler=None, patience=5):
    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    best_state = deepcopy(model.state_dict())
    best_val_acc = -1.0
    patience_left = patience

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
        )

        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_acc)
            else:
                scheduler.step()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = deepcopy(model.state_dict())
            patience_left = patience
        else:
            patience_left -= 1

        if patience_left <= 0:
            print("Early stopping acionado.")
            break

    return history, best_state, best_val_acc

def plot_history(history, title_prefix):
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history["train_loss"], label="train_loss")
    plt.plot(history["val_loss"], label="val_loss")
    plt.title(f"{title_prefix} - Loss")
    plt.xlabel("Época")
    plt.ylabel("Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history["train_acc"], label="train_acc")
    plt.plot(history["val_acc"], label="val_acc")
    plt.title(f"{title_prefix} - Accuracy")
    plt.xlabel("Época")
    plt.ylabel("Accuracy")
    plt.legend()

    plt.tight_layout()
    plt.show()

@torch.no_grad()
def predict_loader(model, loader, device):
    model.eval()
    all_targets = []
    all_preds = []

    for X, y in loader:
        X = X.to(device)
        y = y.to(device)

        logits = model(X)
        preds = torch.argmax(logits, dim=1)

        all_targets.extend(y.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

    return np.array(all_targets), np.array(all_preds)

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

baseline_history, best_state_baseline, best_val_acc = fit_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    epochs=18,
    scheduler=scheduler,
    patience=5,
)

torch.save(best_state_baseline, ARTIFACTS_DIR / "baseline_cnn_best.pt")

In [ ]:
model.load_state_dict(best_state_baseline)

plot_history(baseline_history, "BaselineCNN")

y_true_test_baseline, y_pred_test_baseline = predict_loader(model, test_loader, device)

baseline_acc = accuracy_score(y_true_test_baseline, y_pred_test_baseline)
baseline_p, baseline_r, baseline_f1, _ = precision_recall_fscore_support(
    y_true_test_baseline,
    y_pred_test_baseline,
    pos_label=1,
    average="binary",
    zero_division=0,
)

print(classification_report(y_true_test_baseline, y_pred_test_baseline, target_names=["sem fogo", "com fogo"]))

cm = confusion_matrix(y_true_test_baseline, y_pred_test_baseline)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["sem fogo", "com fogo"])
disp.plot(cmap="Blues")
plt.title("Matriz de confusão - BaselineCNN")
plt.show()

# 8. Modelo 2 — ImprovedCNN

Versão mais profunda da arquitetura, com mais camadas convolucionais e regularização adicional.

In [ ]:
class ImprovedCNN(nn.Module):
    def __init__(self, in_channels=12, num_classes=2):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.15),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.20),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model2 = ImprovedCNN().to(device)
model2

In [ ]:
criterion2 = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer2 = torch.optim.AdamW(model2.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler2 = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer2, mode="max", factor=0.5, patience=2)

improved_history, best_state_improved, best_val_acc_2 = fit_model(
    model=model2,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion2,
    optimizer=optimizer2,
    epochs=24,
    scheduler=scheduler2,
    patience=6,
)

torch.save(best_state_improved, ARTIFACTS_DIR / "improved_cnn_best.pt")

In [ ]:
model2.load_state_dict(best_state_improved)

plot_history(improved_history, "ImprovedCNN")

y_true_test_2, y_pred_test_2 = predict_loader(model2, test_loader, device)

improved_acc = accuracy_score(y_true_test_2, y_pred_test_2)
improved_p, improved_r, improved_f1, _ = precision_recall_fscore_support(
    y_true_test_2,
    y_pred_test_2,
    pos_label=1,
    average="binary",
    zero_division=0,
)

print(classification_report(y_true_test_2, y_pred_test_2, target_names=["sem fogo", "com fogo"]))

cm2 = confusion_matrix(y_true_test_2, y_pred_test_2)
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm2, display_labels=["sem fogo", "com fogo"])
disp2.plot(cmap="Blues")
plt.title("Matriz de confusão - ImprovedCNN")
plt.show()

# 9. Comparação Final dos Modelos

Avaliação consolidada dos experimentos realizados.

In [ ]:
summary_df = pd.DataFrame([
    {
        "Model": "BaselineCNN",
        "Best_Val_Acc": best_val_acc,
        "Test_Accuracy": baseline_acc,
        "Fire_Precision": baseline_p,
        "Fire_Recall": baseline_r,
        "Fire_F1": baseline_f1,
    },
    {
        "Model": "ImprovedCNN",
        "Best_Val_Acc": best_val_acc_2,
        "Test_Accuracy": improved_acc,
        "Fire_Precision": improved_p,
        "Fire_Recall": improved_r,
        "Fire_F1": improved_f1,
    },
])

display(summary_df.sort_values(by="Fire_F1", ascending=False))

plt.figure(figsize=(10, 5))
models = summary_df["Model"]

plt.plot(models, summary_df["Test_Accuracy"], marker="o", label="Accuracy")
plt.plot(models, summary_df["Fire_F1"], marker="o", label="F1 (Fogo)")
plt.plot(models, summary_df["Fire_Recall"], marker="o", label="Recall (Fogo)")

plt.title("Comparação entre Arquiteturas")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "model_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

summary_df.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)
print("Tabela salva.")
print("Gráfico salvo.")

# 10. Demonstração de Inferência

Teste do modelo final em um patch individual do conjunto de teste.

In [ ]:
def predict_single_patch(file_path, model, device, img_size=IMG_SIZE):
    with np.load(file_path, allow_pickle=True) as data:
        image = data["image"]
        label = data["label"]

    x = torch.tensor(image, dtype=torch.float32)
    x = F.interpolate(
        x.unsqueeze(0),
        size=(img_size, img_size),
        mode="bilinear",
        align_corners=False
    ).squeeze(0)

    x = (x - x.mean()) / (x.std() + 1e-6)
    x = x.unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0]
        pred = torch.argmax(probs).item()
        conf = probs[pred].item()

    true_label = int((label > 0).any())
    return true_label, pred, conf

model_final = ImprovedCNN().to(device)
model_final.load_state_dict(best_state_improved)
model_final.eval()

example_file = Path(test_df.iloc[0]["file_path"])
true_label, pred_label, prob = predict_single_patch(example_file, model_final, device)

print("Arquivo:", example_file.name)
print("Rótulo real:", "com fogo" if true_label == 1 else "sem fogo")
print("Predição:", "com fogo" if pred_label == 1 else "sem fogo")
print(f"Confiança: {prob:.2f}")

# 11. Exemplos de Acertos e Erros

Amostras qualitativas da predição do modelo final.

In [ ]:
def predict_file(file_path, model, device, img_size=IMG_SIZE):
    with np.load(file_path, allow_pickle=True) as data:
        image = data["image"]
        label = data["label"]

    x = torch.tensor(image, dtype=torch.float32)
    x = F.interpolate(
        x.unsqueeze(0),
        size=(img_size, img_size),
        mode="bilinear",
        align_corners=False
    ).squeeze(0)

    x = (x - x.mean()) / (x.std() + 1e-6)
    x = x.unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(x)
        pred = torch.argmax(logits, dim=1).item()

    true_label = int((label > 0).any())
    return true_label, pred, image, label

rows = []
for _, row in test_df.iterrows():
    true_label, pred_label, image, label = predict_file(Path(row["file_path"]), model_final, device)
    rows.append({
        "file_path": row["file_path"],
        "true_label": true_label,
        "pred_label": pred_label,
        "correct": int(true_label == pred_label),
        "image": image,
        "mask": label
    })

test_results_df = pd.DataFrame(rows)

correct_df = test_results_df[test_results_df["correct"] == 1]
wrong_df = test_results_df[test_results_df["correct"] == 0]
false_positive_df = test_results_df[(test_results_df["true_label"] == 0) & (test_results_df["pred_label"] == 1)]
false_negative_df = test_results_df[(test_results_df["true_label"] == 1) & (test_results_df["pred_label"] == 0)]

print("Acertos:", len(correct_df))
print("Erros:", len(wrong_df))
print("Falsos positivos:", len(false_positive_df))
print("Falsos negativos:", len(false_negative_df))

def show_examples(df_subset, title, n=2):
    if len(df_subset) == 0:
        print(f"Sem exemplos para: {title}")
        return

    n = min(n, len(df_subset))
    chosen = df_subset.sample(n=n, random_state=42)

    fig, axes = plt.subplots(n, 2, figsize=(10, 4 * n))
    if n == 1:
        axes = np.array([axes])

    for i, (_, row) in enumerate(chosen.iterrows()):
        rgb = make_rgb(row["image"])
        mask = (row["mask"] > 0).astype(np.uint8) * 255

        axes[i, 0].imshow(rgb)
        axes[i, 0].set_title(
            f"Real: {'com fogo' if row['true_label'] else 'sem fogo'} | "
            f"Previsto: {'com fogo' if row['pred_label'] else 'sem fogo'}"
        )
        axes[i, 0].axis("off")

        axes[i, 1].imshow(mask, cmap="gray")
        axes[i, 1].set_title("Máscara real")
        axes[i, 1].axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

show_examples(correct_df[correct_df["true_label"] == 0], "Acertos: sem fogo", n=2)
show_examples(correct_df[correct_df["true_label"] == 1], "Acertos: com fogo", n=2)
show_examples(false_positive_df, "Erros: falso positivo", n=2)
show_examples(false_negative_df, "Erros: falso negativo", n=2)

# 12. Conclusão

Foram avaliadas duas arquiteturas de CNN criadas do zero para classificação de imagens de satélite no contexto de monitoramento de queimadas.

Resultados principais:

- BaselineCNN: desempenho inicial satisfatório.
- ImprovedCNN: melhor resultado geral, com acurácia de teste maior.
- A meta de 88% de acurácia no teste não foi atingida, principalmente por causa do desbalanceamento do dataset e da maior dificuldade da classe minoritária ("com fogo").

O modelo escolhido como solução final foi a ImprovedCNN, por apresentar o melhor equilíbrio entre acurácia, precisão, recall e F1-score.

## Limitações e Trabalhos Futuros

Limitações:

- Dataset desbalanceado.
- Número limitado de cenas disponíveis.
- Classificação baseada apenas na presença ou ausência de fogo.

Possíveis melhorias:

- Data augmentation adicional.
- Ajuste fino de hiperparâmetros.
- Arquiteturas mais avançadas.
- Segmentação semântica para localização precisa das áreas queimadas.